# Q12.
```{admonition}
:class: note
Consider the RNN fit to the `NYSE` data. Modify the code to allow inclusion of the variable `day_of_week`, and fit the RNN. Compute the test $R^{2}$.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

In [3]:
import torch
from torchinfo import summary
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
nyse = pd.read_csv('../../../ALL CSV FILES - 2nd Edition/NYSE.csv', parse_dates=['date']).sort_index().drop(columns=['train'])

In [5]:
nyse_train, nyse_test = train_test_split(nyse, shuffle=False, train_size=4276)

In [6]:
scaler = StandardScaler()
scale_cols = ['DJ_return','log_volume','log_volatility']
nyse_train[scale_cols] = scaler.fit_transform(nyse_train[scale_cols])
nyse_test[scale_cols] = scaler.transform(nyse_test[scale_cols])
nyse_scaled = pd.concat([nyse_train,nyse_test])
nyse_scaled = pd.get_dummies(nyse_scaled['day_of_week']).join(nyse_scaled)
nyse_scaled = nyse_scaled.set_index('date').drop(columns=['day_of_week'])

In [7]:
nyse_scaled.head(3)

,fri,mon,thur,tues,wed,DJ_return,log_volume,log_volatility
date,,,,,,,,
1962-12-03,False,True,False,False,False,-0.559615,0.193763,-3.982432
1962-12-04,False,False,False,True,False,0.959704,1.552668,-2.240505
1962-12-05,False,False,False,False,True,0.468531,2.328698,-2.134713


In [8]:
nyse_lag = pd.concat([nyse_scaled.shift(lag).add_suffix(f'_lag{lag}') for lag in range(5,0,-1)]+[nyse_scaled], axis=1).dropna()
nyse_lag.head(3)

,fri_lag5,mon_lag5,thur_lag5,tues_lag5,wed_lag5,DJ_return_lag5,log_volume_lag5,log_volatility_lag5,fri_lag4,mon_lag4,...,log_volume_lag1,log_volatility_lag1,fri,mon,thur,tues,wed,DJ_return,log_volume,log_volatility
date,,,,,,,,,,,,,,,,,,,,,
1962-12-10,False,True,False,False,False,-0.559615,0.193763,-3.982432,False,False,...,0.244084,-2.213740,False,True,False,False,False,-1.347250,0.629963,-1.132250
1962-12-11,False,False,False,True,False,0.959704,1.552668,-2.240505,False,False,...,0.629963,-1.132250,False,False,False,True,False,0.007932,0.002680,-1.265313
1962-12-12,False,False,False,False,True,0.468531,2.328698,-2.134713,False,False,...,0.002680,-1.265313,False,False,False,False,True,0.408248,0.059592,-1.309001


In [9]:
Y = nyse_lag['log_volume'].to_numpy().astype(np.float32)
X = nyse_lag.drop(columns=nyse_scaled.columns).to_numpy().astype(np.float32)
X_rnn = X.reshape(-1,5,8)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_rnn, Y, shuffle=False, train_size=4276)

In [11]:
torch.manual_seed(1728)

class NYSEModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(8,12,batch_first=True)
        self.dense = nn.Linear(12,1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        val, h_n = self.rnn(x)
        return self.dense(self.dropout(val[:,-1]))

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_t = torch.tensor(X_train,dtype=torch.float32)
y_train_t = torch.tensor(y_train,dtype=torch.float32).unsqueeze(1)
nyse_train = TensorDataset(X_train_t,y_train_t)

train_loader = DataLoader(
    nyse_train,
    batch_size=64,
    shuffle=False
)

X_test_t = torch.tensor(X_test,dtype=torch.float32)
y_test_t = torch.tensor(y_test,dtype=torch.float32)

In [13]:
nyse_model = NYSEModel().to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(nyse_model.parameters(),lr=0.001)
epochs = 100

for epoch in range(epochs):
    nyse_model.train()

    for X_batch, y_batch in train_loader:
        mses = nyse_model(X_batch.to(device))
        loss = criterion(mses,y_batch.to(device))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    nyse_model.eval()
    with torch.no_grad():
        preds = nyse_model(X_test_t.to(device))
        mse = mean_squared_error(y_test,preds.cpu().numpy())
        r2 = r2_score(y_test,preds.cpu().numpy())
    if (epoch+1) % 10 == 0:
        print(f'Epoch {epoch+1}: mse={mse:.4f}, r2={r2:.4f}')

Epoch 10: mse=0.6168, r2=0.4289
Epoch 20: mse=0.6074, r2=0.4376
Epoch 30: mse=0.5984, r2=0.4459
Epoch 40: mse=0.5921, r2=0.4518
Epoch 50: mse=0.5903, r2=0.4535
Epoch 60: mse=0.5915, r2=0.4523
Epoch 70: mse=0.5907, r2=0.4531
Epoch 80: mse=0.5933, r2=0.4506
Epoch 90: mse=0.5916, r2=0.4522
Epoch 100: mse=0.5917, r2=0.4521


In [15]:
nyse_model.eval()

with torch.no_grad():
    pred = nyse_model(X_test_t.to(device))
    mse = mean_squared_error(y_test,preds.cpu().numpy())
    r2 = r2_score(y_test,preds.cpu().numpy())
print(f'Test MSE: {mse:.4f}')
print(f'Test R2 score: {r2:.4f}')

Test MSE: 0.5917
Test R2 score: 0.4521
